In [1]:
import os, sys
sys.path.append('..')
from dotenv import load_dotenv
import anthropic
from utils.helpers import CLAUDE_SONNET, CLAUDE_HAIKU, CLAUDE_OPUS, DEFAULT_MODEL

load_dotenv()
client = anthropic.Anthropic()
MODEL = DEFAULT_MODEL

def ask(prompt: str, system: str = "", max_tokens: int = 512) -> str:
    kwargs = {"model": MODEL, "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[0].text


## 1. Ser Claro y Directo

In [ ]:
if "client" not in dir():
    import sys; sys.path.append('..')
    from dotenv import load_dotenv
    import anthropic
    from utils.helpers import DEFAULT_MODEL
    load_dotenv()
    client = anthropic.Anthropic()
    MODEL = DEFAULT_MODEL
    def ask(prompt: str, system: str = "", max_tokens: int = 512) -> str:
        kwargs = {"model": MODEL, "max_tokens": max_tokens,
                  "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        return client.messages.create(**kwargs).content[0].text

# Malo: vago
prompt_vago = "Háblame sobre Python"

# Bueno: claro y directo
prompt_claro = """Explica las 3 características principales de Python que lo hacen 
ideal para principiantes. Usa viñetas y ejemplos de código cortos."""

print("=== Prompt vago ===")
print(ask(prompt_vago)[:300])

print("\n=== Prompt claro y directo ===")
print(ask(prompt_claro)[:500])


## 2. Ser Específico

In [ ]:
# ❌ Poco específico
prompt_generico = "Escribe un email sobre el proyecto."

# ✅ Específico con contexto
prompt_especifico = """
Escribe un email profesional con las siguientes especificaciones:
- De: Ana López (Gerente de Proyecto)
- Para: El equipo de desarrollo (8 personas)
- Asunto: Actualización del proyecto AppMobile v2.0
- Contenido: 
  * El proyecto va 2 semanas adelantado
  * La demo con el cliente es el viernes 10 de abril a las 3pm
  * Todos deben preparar un resumen de 2 minutos de su módulo
- Tono: profesional pero cercano
- Longitud: máximo 200 palabras
"""

print("=== Específico ===")
print(ask(prompt_especifico))

## 3. Estructurar con XML Tags

In [ ]:
# Los XML tags ayudan a Claude a diferenciar partes del prompt
# Especialmente útil con documentos, contexto largo o múltiples inputs

documento = """
El cambio climático es uno de los mayores desafíos del siglo XXI. 
Las temperaturas globales han aumentado 1.1°C desde la era preindustrial.
Los principales causantes son las emisiones de CO2 por combustibles fósiles.
Los efectos incluyen: aumento del nivel del mar, eventos climáticos extremos 
y pérdida de biodiversidad.
"""

prompt_con_xml = """
<documento>
{doc}
</documento>

<instrucciones>
Basándote ÚNICAMENTE en el documento proporcionado:
1. Identifica el tema principal (1 oración)
2. Lista los datos numéricos mencionados
3. Lista las consecuencias mencionadas
</instrucciones>

<formato>
Responde con secciones claramente separadas por los mismos headers.
</formato>
""".format(doc=documento)

print(ask(prompt_con_xml))

In [ ]:
# XML para múltiples documentos
doc1 = "Python es un lenguaje interpretado de alto nivel, creado por Guido van Rossum en 1991."
doc2 = "JavaScript fue creado por Brendan Eich en 1995 y es el lenguaje de la web."

prompt_multi_doc = """
<documentos>
  <documento id="1">\n{d1}\n  </documento>
  <documento id="2">\n{d2}\n  </documento>
</documentos>

<pregunta>¿Qué tienen en común estos dos lenguajes de programación?</pregunta>

<restriccion>Menciona SOLO información presente en los documentos.</restriccion>
""".format(d1=doc1, d2=doc2)

print(ask(prompt_multi_doc))

## 4. Few-Shot Prompting (Proveer Ejemplos)

In [ ]:
# Zero-shot (sin ejemplos)
prompt_zeroshot = """
Convierte el siguiente precio de formato texto a número:
"veinte y cinco dólares con cincuenta centavos"
"""

# Few-shot (con ejemplos)
prompt_fewshot = """
Convierte precios de texto a número decimal. Ejemplos:

Texto: "diez dólares" → 10.00
Texto: "cinco dólares con veinticinco centavos" → 5.25
Texto: "cien dólares con cincuenta centavos" → 100.50
Texto: "doce dólares con noventa y nueve centavos" → 12.99

Ahora convierte:
Texto: "veinte y cinco dólares con cincuenta centavos" →
"""

print("Zero-shot:", ask(prompt_zeroshot, max_tokens=50))
print("Few-shot: ", ask(prompt_fewshot, max_tokens=20))

In [ ]:
# Few-shot para clasificación con formato específico
prompt_clasificacion = """
Clasifica tickets de soporte técnico en: HARDWARE, SOFTWARE, RED, o USUARIO.

<ejemplos>
<ejemplo>
<ticket>Mi computadora no enciende, hace ruidos extraños</ticket>
<categoria>HARDWARE</categoria>
</ejemplo>
<ejemplo>
<ticket>El programa Excel se cierra solo al abrir archivos grandes</ticket>
<categoria>SOFTWARE</categoria>
</ejemplo>
<ejemplo>
<ticket>No puedo acceder a internet pero mi compañero sí puede</ticket>
<categoria>RED</categoria>
</ejemplo>
<ejemplo>
<ticket>Olvidé mi contraseña de Windows</ticket>
<categoria>USUARIO</categoria>
</ejemplo>
</ejemplos>

Clasifica este ticket:
<ticket>La impresora no aparece en la lista de dispositivos disponibles</ticket>

Responde SOLO con la categoría.
"""

print("Categoría:", ask(prompt_clasificacion, max_tokens=20))

## 5. Chain of Thought (Razonamiento Paso a Paso)

In [ ]:
problema = """
En una tienda, un libro cuesta $15. Si compras 3 libros o más, obtienes 20% de descuento
en todos los libros. ¿Cuánto pagas por 5 libros?
"""

# Sin Chain of Thought
print("=== Sin CoT ===")
print(ask(problema + "\nResponde solo con el precio final.", max_tokens=30))

# Con Chain of Thought
print("\n=== Con Chain of Thought ===")
prompt_cot = problema + """
Resuelve paso a paso:
1. Precio base
2. ¿Aplica descuento?
3. Cálculo del descuento
4. Precio final
"""
print(ask(prompt_cot, max_tokens=200))

## 6. Ejercicio Final

**Tarea**: Construye un prompt que analice un CV y extraiga información clave en JSON.
Aplica: claridad, especificidad, XML tags y ejemplos.

In [ ]:
cv_ejemplo = """
JUAN PÉREZ MARTÍNEZ
Email: juan.perez@email.com | Tel: +34 612 345 678
Madrid, España

EXPERIENCIA:
2021-presente: Senior Developer en TechCorp S.A.
  - Lideré equipo de 5 desarrolladores
  - Implementé microservicios con Python y Docker

2018-2021: Developer en StartupXYZ
  - Desarrollé APIs REST con Node.js
  - Mantenimiento de bases de datos PostgreSQL

EDUCACIÓN:
2014-2018: Grado en Ingeniería Informática - Universidad Complutense de Madrid

HABILIDADES: Python, JavaScript, Docker, PostgreSQL, Git, Agile/Scrum
"""

import json

prompt_cv = """
<cv>
{cv}
</cv>

<tarea>
Extrae la información del CV en formato JSON con esta estructura exacta:
{{
  "nombre": string,
  "contacto": {{"email": string, "telefono": string, "ubicacion": string}},
  "experiencia_anos": number,
  "empresas": [string],
  "habilidades": [string],
  "nivel": "junior" | "mid" | "senior"
}}
</tarea>

<restriccion>Solo JSON válido, sin texto adicional.</restriccion>
""".format(cv=cv_ejemplo)

resp = ask(prompt_cv, max_tokens=400)
print("JSON extraído:")
try:
    data = json.loads(resp)
    print(json.dumps(data, ensure_ascii=False, indent=2))
except:
    print(resp)